# BirdCLEF 2026 — Perch Submit v4
**Strictly follows SUBMISSION_MASTER_GUIDE.md (0.746 baseline).**
Perch ONNX logits -> taxonomy mapping -> BirdCLEF 234 species.


In [ ]:
# [1] Install onnxruntime from local wheel
import subprocess, sys, os, glob
def _fd(p):
    d=f"/kaggle/input/{p}"
    if os.path.isdir(d): return d
    for r,_,_ in os.walk("/kaggle/input"):
        if p in r: return r
    return d
WD=_fd("birdclef-perch-models")
try:
    import onnxruntime
except ImportError:
    wh=sorted(glob.glob(os.path.join(WD,"*.whl")))
    if wh:
        subprocess.check_call([sys.executable,"-m","pip","install","--no-deps",wh[0]],
            stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
        import onnxruntime
print(f"onnxruntime {onnxruntime.__version__} OK")


In [ ]:
# [2] Imports + Paths (GUIDE Section 5.1)
from pathlib import Path
import os, sys, time, re
import numpy as np, pandas as pd
import onnxruntime as ort
import librosa

INPUT = Path('/kaggle/input')
WORK  = Path('/kaggle/working')

# Find competition dataset (GUIDE: check both locations)
COMP = None
for c in [INPUT/'birdclef-2026', INPUT/'competitions'/'birdclef-2026']:
    if c.exists() and (c/'sample_submission.csv').exists():
        COMP = c; break
assert COMP is not None, 'Competition dataset not found'
print(f'COMP: {COMP}')

SAMPLE_SUB = COMP / 'sample_submission.csv'
TEST_DIR   = COMP / 'test_soundscapes'
TAXONOMY   = COMP / 'taxonomy.csv'
SUBM_OUT   = WORK / 'submission.csv'

# Find Perch model dataset
MODEL_DIR = None
for root, dirs, files in os.walk(INPUT):
    if 'perch_v2.onnx' in files:
        MODEL_DIR = Path(root); break
assert MODEL_DIR is not None, 'Perch ONNX not found'
print(f'MODEL_DIR: {MODEL_DIR}')

PERCH_ONNX    = MODEL_DIR / 'perch_v2.onnx'
PERCH_LABELS  = MODEL_DIR / 'labels.csv'

SR, DURATION = 32000, 5
SEGMENT_SAMPLES = SR * DURATION
BATCH_SIZE = 32

sopts = ort.SessionOptions()
sopts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sopts.intra_op_num_threads = 8


In [ ]:
# [3] Load: Perch ONNX, taxonomy, sample_submission
perch_sess = ort.InferenceSession(str(PERCH_ONNX), sopts, providers=["CPUExecutionProvider"])
perch_in   = perch_sess.get_inputs()[0].name
perch_out  = [o.name for o in perch_sess.get_outputs()]
print(f"Perch outputs: {perch_out}")

perch_labels_df = pd.read_csv(PERCH_LABELS)
taxonomy_df     = pd.read_csv(TAXONOMY)
sample          = pd.read_csv(SAMPLE_SUB)

SPECIES_COLS = [c for c in sample.columns if c != 'row_id']
N_SPECIES    = len(SPECIES_COLS)
species_to_idx = {sp: i for i, sp in enumerate(SPECIES_COLS)}

print(f"Perch classes: {len(perch_labels_df)}")
print(f"Taxonomy: {len(taxonomy_df)}")
print(f"Species: {N_SPECIES}")
print(f"Sample rows: {len(sample)}")


In [ ]:
# [4] Build Perch -> BirdCLEF mapping via taxonomy
tax_id_to_sci = {}
tax_id_to_common = {}
for _, row in taxonomy_df.iterrows():
    pid = str(row["primary_label"])
    tax_id_to_sci[pid] = str(row.get("scientific_name","")).lower().strip()
    tax_id_to_common[pid] = str(row.get("common_name","")).lower().strip()

perch_labels_list = []
for i, row in perch_labels_df.iterrows():
    lbl = str(row.get("label", row.get("scientific_name",""))).lower().strip()
    perch_labels_list.append(lbl)

bc_to_perch = {}
perch_to_bc = {}

for bc_sp in SPECIES_COLS:
    bc_idx = species_to_idx[bc_sp]
    sci = tax_id_to_sci.get(bc_sp, bc_sp.lower().replace("_"," ")).strip()
    common = tax_id_to_common.get(bc_sp,"").strip()
    matched = False
    # 1) exact science name
    for pi, pl in enumerate(perch_labels_list):
        if pl == sci:
            bc_to_perch[bc_idx]=pi; perch_to_bc.setdefault(pi,[]).append(bc_idx); matched=True; break
    # 2) genus
    if not matched and " " in sci:
        genus = sci.split()[0]
        for pi, pl in enumerate(perch_labels_list):
            if pl.startswith(genus+" "):
                bc_to_perch[bc_idx]=pi; perch_to_bc.setdefault(pi,[]).append(bc_idx); matched=True; break
    # 3) exact common name
    if not matched and common:
        for pi, pl in enumerate(perch_labels_list):
            if pl == common:
                bc_to_perch[bc_idx]=pi; perch_to_bc.setdefault(pi,[]).append(bc_idx); matched=True; break
    # 4) substring
    if not matched and common:
        for pi, pl in enumerate(perch_labels_list):
            if common in pl or pl in common:
                bc_to_perch[bc_idx]=pi; perch_to_bc.setdefault(pi,[]).append(bc_idx); matched=True; break

matched = len(bc_to_perch)
print(f"Matched: {matched}/{N_SPECIES}")
if matched < N_SPECIES:
    unm = [sp for sp in SPECIES_COLS if species_to_idx[sp] not in bc_to_perch]
    print(f"Unmatched ({len(unm)}): {unm[:15]}")


In [ ]:
# [5] Perch inference functions

def load_segments(file_path, seg_s=5.0, target_sr=32000):
    """Load audio and split into non-overlapping segments."""
    y, sr = librosa.load(file_path, sr=None, mono=True)
    if sr != target_sr:
        y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)
    slen = int(seg_s * target_sr)
    if len(y) < slen:
        y = np.pad(y, (0, slen - len(y)))
    segs = []
    for st in range(0, len(y) - slen + 1, slen):
        segs.append(y[st:st+slen])
    if not segs:
        segs.append(y[:slen])
    return [s.astype(np.float32) for s in segs]


def predict_segments(waveforms):
    """Run Perch ONNX on a list of waveforms -> (N_seg, N_SPECIES) probs."""
    wavs = np.stack(waveforms)  # (N, 160000)
    n = len(wavs)
    all_probs = np.zeros((n, N_SPECIES), dtype=np.float32)
    for i in range(0, n, BATCH_SIZE):
        batch = wavs[i:i+BATCH_SIZE]
        outs = perch_sess.run(perch_out, {perch_in: batch})
        od = dict(zip(perch_out, outs))
        logits = None
        for k in ["label", "logits"]:
            if k in od: logits = od[k]; break
        if logits is None: continue
        p = 1.0 / (1.0 + np.exp(-logits))
        bc_p = np.zeros((len(batch), N_SPECIES), dtype=np.float32)
        for bc_idx, pi in bc_to_perch.items():
            if pi < p.shape[1]:
                bc_p[:, bc_idx] = p[:, pi]
        all_probs[i:i+len(batch)] = bc_p
    return all_probs


In [ ]:
# [6] Inference + Submission (GUIDE Section 5.6)
ogg_files = sorted(TEST_DIR.glob('*.ogg')) if TEST_DIR.exists() else []
print(f'Test soundscapes (.ogg): {len(ogg_files)}')

if not ogg_files:
    # === DRY-RUN: copy sample as-is (GUIDE Section 3 & 5.6) ===
    print('WARNING: No test soundscapes. Copying sample_submission.csv as-is.')
    sample.to_csv(SUBM_OUT, index=False)
    print(f'Dry-run submission written: {len(sample)} rows')

else:
    # === REAL SCORING: process all .ogg files ===
    submission_rows = {}  # {row_id: [prob_0, prob_1, ...]}

    t0 = time.time()
    for idx, ogg_path in enumerate(ogg_files):
        try:
            segments = load_segments(str(ogg_path), seg_s=DURATION)
        except Exception as e:
            print(f'  [ERR] {ogg_path.name}: {e}')
            continue
        if not segments:
            continue

        probs = predict_segments(segments)  # (N_seg, N_SPECIES)
        stem = ogg_path.stem  # e.g. BC2026_Test_0001_S05_20250227_010002

        for seg_idx in range(len(segments)):
            end_sec = (seg_idx + 1) * DURATION
            row_id = f"{stem}_{end_sec}"
            submission_rows[row_id] = probs[seg_idx].tolist()

        if (idx + 1) % 50 == 0 or idx < 3:
            ela = time.time() - t0
            eta = ela/(idx+1)*len(ogg_files)
            print(f'  [{idx+1:3d}/{len(ogg_files)}] {ogg_path.name} '
                  f'({len(segments)} seg) {ela:.0f}s/~{eta:.0f}s')

    print(f'Inference done: {len(ogg_files)} files in {time.time()-t0:.0f}s')

    # Align with sample_submission row_ids (GUIDE Section 5.6)
    rows = []
    for _, sample_row in sample.iterrows():
        row_id = sample_row['row_id']
        row_data = {'row_id': row_id}
        if row_id in submission_rows:
            for cls, prob in zip(SPECIES_COLS, submission_rows[row_id]):
                row_data[cls] = float(prob)
        else:
            for cls in SPECIES_COLS:
                row_data[cls] = 0.0
        rows.append(row_data)

    sub = pd.DataFrame(rows)
    # CRITICAL: force exact column order from sample (GUIDE Section 5.6)
    sub = sub[['row_id'] + SPECIES_COLS]
    for col in SPECIES_COLS:
        sub[col] = sub[col].astype(float)

    sub.to_csv(SUBM_OUT, index=False, float_format='%.6f')
    print(f'Submission written: {len(sub)} rows')


In [ ]:
# [7] Validation (GUIDE Section 6 — mandatory)
sample = pd.read_csv(SAMPLE_SUB)
sub    = pd.read_csv(SUBM_OUT)

errors = []

# 1. Shape
if sub.shape != sample.shape:
    errors.append(f"Shape mismatch: {sub.shape} vs {sample.shape}")

# 2. Columns (order AND names)
if list(sub.columns) != list(sample.columns):
    errors.append("Column mismatch")

# 3. Row IDs
if not sub['row_id'].equals(sample['row_id']):
    errors.append("row_id mismatch")

# 4. No NaN
proba = sub.drop(columns=['row_id'])
if proba.isna().sum().sum() != 0:
    errors.append(f"NaN values: {proba.isna().sum().sum()}")

# 5. Values in [0, 1]
if not ((proba >= 0.0) & (proba <= 1.0)).all().all():
    errors.append("Values out of [0,1] range")

# 6. Species column order
if list(proba.columns) != SPECIES_COLS:
    errors.append("Species column order mismatch")

if errors:
    for e in errors:
        print(f"FAIL: {e}")
    raise RuntimeError("Submission validation FAILED")
else:
    print(f"VALID: {len(sub)} rows x {len(sub.columns)} cols")
    print(f"  Row IDs match sample: {sub['row_id'].equals(sample['row_id'])}")
    print(f"  Range: [{proba.min().min():.6f}, {proba.max().max():.6f}]")
    print(f"  Mean: {proba.mean().mean():.6f}")
    print(f"  Active species (>0.01): {(proba.max(axis=0) > 0.01).sum()}/{N_SPECIES}")


In [ ]:
# [8] Done
print("\n" + "="*50)
print("SUBMISSION READY — perch_submit_v4")
print("="*50)
print(f"Species matched: {matched}/{N_SPECIES}")
print(f"Soundscapes processed: {len(ogg_files)}")
runtime = time.time() - t0 if 't0' in dir() else 0
print(f"Runtime: {runtime:.0f}s")
print(f"Output: {SUBM_OUT}")
print("\nSubmit to competition!")
